# 07 — Validate routing outputs

Validate source coverage, graph status, supported schemas, non-negative and nested car/pedestrian accessibility contours, PT zero-stop flags, lagged-stock provenance, and cumulative firm accessibility against the long Fachgruppe table.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import duckdb
import numpy as np
import pandas as pd


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or a subdirectory containing ANAL/ and OGD/.")

PROJECT_DIR = discover_project_dir()
# Set this to the same value used in steps 01, 03, and 06.
CELL_SIZE = 100  # Metres; set the same value in steps 01, 03, and 06, 08–10.
RESOLUTION = f"{CELL_SIZE}m"
ANAL_DATA = PROJECT_DIR / "ANAL" / "data" / RESOLUTION
ROUTING_DATA = ANAL_DATA / "routing"
SHARED_ROUTING_DATA = PROJECT_DIR / "ANAL" / "data" / "routing"
POI_DIR = SHARED_ROUTING_DATA / "destinations"
PBF_DIR = PROJECT_DIR / "OGD" / "GIP"
PANEL_PATH = ANAL_DATA / f"raster_quarter_panel_{RESOLUTION}.parquet"
ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / f"active_routing_cells_{RESOLUTION}.parquet"
STATUS_PATH = ROUTING_DATA / "status" / "routing_feature_status.csv"
GRAPH_STATUS_PATH = SHARED_ROUTING_DATA / "status" / "graph_build_status.csv"
SKIPPED_ACCESSIBILITY_PATH = ROUTING_DATA / "status" / "accessibility_skipped_origins.csv"
REPORT_PATH = ROUTING_DATA / "reports" / "routing_validation_summary.csv"
WORK_ROOT = ROUTING_DATA / "work" / "routing_features"
GRAPH_VALHALLA_IMAGE = "ghcr.io/valhalla/valhalla-scripted:3.8.3"
GRAPH_VALHALLA_IMAGE_ID = "sha256:24ef7955899dececb94e26c6dfb89d64fabfae875f980432694b0261eb6c251b"
YEARS = range(2015, 2026)
sys.path.insert(0, str(PROJECT_DIR / "ANAL" / "routing"))
from routing_utils import ACCESS_MINUTES, fachgruppe_ids, main_access_columns
print(f"Resolution: {RESOLUTION}; validation folder: {ROUTING_DATA}")
FACHGRUPPE_IDS = fachgruppe_ids(PANEL_PATH)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def result(year, product, check, passed):
    return {"year": year, "product": product, "check": check, "status": "OK" if bool(passed) else "CHECK"}


def record(year, product, check, passed):
    records.append(result(year, product, check, passed))
    return bool(passed)


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def open_product(name, path, required, year, product):
    if not record(year, product, "canonical output exists", path.exists()):
        return False
    parquet_path = path / "**" / "*.parquet" if path.is_dir() else path
    con.execute(f"CREATE OR REPLACE TEMP VIEW {name} AS SELECT * FROM read_parquet({sql_string(parquet_path.as_posix())}, hive_partitioning=true)")
    available = {row[0] for row in con.execute(f"DESCRIBE {name}").fetchall()}
    missing = set(required) - available
    return record(year, product, "required schema" + (f": missing {sorted(missing)}" if missing else ""), not missing)


def query_ok(year, product, check, query):
    return record(year, product, check, con.execute(query).fetchone()[0])


def check_keys(view, expected, keys, year, product):
    columns = ", ".join(keys)
    unique = query_ok(year, product, "unique observation keys", f"SELECT count(*) = count(DISTINCT ({columns})) FROM {view}")
    complete = query_ok(year, product, "exact expected observation coverage", f"""
        SELECT NOT EXISTS (
            (SELECT {columns} FROM {view} EXCEPT SELECT {columns} FROM {expected})
            UNION ALL
            (SELECT {columns} FROM {expected} EXCEPT SELECT {columns} FROM {view})
        )
    """)
    return unique and complete


def numeric_checks(view, columns, year, product):
    # A single columnar scan; documented isochrone skips retain their missing masses.
    fields = []
    for column in columns:
        value = f'TRY_CAST("{column}" AS DOUBLE)'
        fields.append(f"count(*) FILTER (WHERE {value} IS NULL OR NOT isfinite({value}) OR {value} < 0)")
    invalid = con.execute(f"SELECT {', '.join(fields)} FROM {view} WHERE grid_id NOT IN (SELECT grid_id FROM skipped)").fetchone()
    for column, count in zip(columns, invalid):
        record(year, product, f"{column} finite and non-negative", count == 0)


def check_contours(view, prefixes, year, product):
    comparisons = [(f"{prefix}_{a}min", f"{prefix}_{b}min")
        for prefix, minutes in prefixes.items() for a, b in zip(minutes, minutes[1:])]
    fields = [f'count(*) FILTER (WHERE "{a}" > "{b}" + 1e-8)' for a, b in comparisons]
    counts = con.execute(f"SELECT {', '.join(fields)} FROM {view} WHERE grid_id NOT IN (SELECT grid_id FROM skipped)").fetchone()
    for (a, b), count in zip(comparisons, counts):
        record(year, product, f"{a} <= {b}", count == 0)


def check_period(view, year, product):
    query_ok(year, product, "period matches year and quarter", f"""SELECT count(*) = 0 FROM {view}
        WHERE period IS DISTINCT FROM CAST(year AS VARCHAR) || 'Q' || CAST(quarter AS VARCHAR)""")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


records = []
firms_path = ANAL_DATA / f"firms_assigned_{RESOLUTION}.geoparquet"
missing_sources = [p for p in [PANEL_PATH, ACTIVE_CELLS_PATH, firms_path] if not p.exists()]
for path in [PANEL_PATH, ACTIVE_CELLS_PATH, firms_path]:
    record("all", "source", f"{path.name} exists", path.exists())
if missing_sources:
    pd.DataFrame(records).to_csv(REPORT_PATH, index=False)
    raise FileNotFoundError(f"Missing source inputs: {missing_sources}")

skipped_accessibility = pd.read_csv(SKIPPED_ACCESSIBILITY_PATH) if SKIPPED_ACCESSIBILITY_PATH.exists() else pd.DataFrame(columns=["year", "grid_id"])
skip_fields = {"year", "grid_id", "lat", "lon", "reason"}
skip_valid = (SKIPPED_ACCESSIBILITY_PATH.exists() and skip_fields.issubset(skipped_accessibility)
              and skipped_accessibility[list(skip_fields)].notna().all().all()
              and not skipped_accessibility.duplicated(["year", "grid_id"]).any())
record("all", "manifest", "documented skips have complete unique metadata", skip_valid)
if not {"year", "grid_id"}.issubset(skipped_accessibility):
    skipped_accessibility = pd.DataFrame(columns=["year", "grid_id"])

con = duckdb.connect()
con.execute("SET memory_limit = '2GB'")
con.execute(f"SET temp_directory = {sql_string((ROUTING_DATA / 'work' / '.validation_tmp').as_posix())}")
try:
    origin_ok = open_product("origins", ACTIVE_CELLS_PATH, ["grid_id"], "all", "source")
    source_ok = open_product("source_firms", firms_path,
        ["firm_id", "grid_id", "Fachgruppe_ID", "founding_date", "exit_date", "exit_observed"], "all", "source")
    if not origin_ok or not source_ok:
        raise ValueError("Missing source columns; see the validation report.")
    query_ok("all", "source", "nonempty unique origin IDs", "SELECT count(*) > 0 AND count(*) = count(DISTINCT grid_id) AND count(grid_id) = count(*) FROM origins")
    query_ok("all", "source", "unique complete firm identifiers", "SELECT count(*) = count(DISTINCT firm_id) AND count(grid_id) = count(*) FROM source_firms")
    query_ok("all", "source", "normalized dates and exit flags consistent", """SELECT count(*) = 0 FROM source_firms
        WHERE (founding_date IS NOT NULL AND TRY_CAST(founding_date AS TIMESTAMP) IS NULL)
           OR (exit_date IS NOT NULL AND TRY_CAST(exit_date AS TIMESTAMP) IS NULL)
           OR exit_date < founding_date
           OR TRY_CAST(exit_observed AS BOOLEAN) IS DISTINCT FROM (exit_date IS NOT NULL)""")
    con.register("configured_groups", pd.DataFrame({"Fachgruppe_ID": FACHGRUPPE_IDS}))
    record("all", "source", "configured Fachgruppe IDs are nonempty and unique", bool(FACHGRUPPE_IDS) and len(set(FACHGRUPPE_IDS)) == len(FACHGRUPPE_IDS))

    for year in YEARS:
        print(f"Validate {year}: scanning Parquet products", flush=True)
        selected_skips = skipped_accessibility.loc[pd.to_numeric(skipped_accessibility.year, errors="coerce").eq(year), ["grid_id"]].copy()
        selected_skips["grid_id"] = selected_skips.grid_id.astype(str)
        con.register("skipped", selected_skips)
        query_ok(year, "manifest", "documented skipped origins belong to this run", "SELECT NOT EXISTS (SELECT grid_id FROM skipped EXCEPT SELECT grid_id FROM origins)")
        con.execute(f"CREATE OR REPLACE TEMP VIEW expected_nearest AS SELECT grid_id, {year} AS year FROM origins")
        con.execute(f"CREATE OR REPLACE TEMP VIEW expected_grid AS SELECT grid_id, {year} AS year, q.quarter FROM origins CROSS JOIN range(1, 5) q(quarter)")
        # Independent event-window coverage, including firms exiting at quarter start.
        con.execute(f"""CREATE OR REPLACE TEMP VIEW expected_firms AS
            SELECT f.firm_id, f.grid_id, CAST(f.Fachgruppe_ID AS VARCHAR) AS Fachgruppe_ID,
                   {year} AS year, q.quarter,
                   (f.founding_date < q.start_date AND coalesce(f.exit_date, TIMESTAMP 'infinity') >= q.start_date) AS included_in_lagged_stock
            FROM source_firms f CROSS JOIN (
                SELECT quarter, make_date({year}, 3 * quarter - 2, 1) AS start_date,
                       make_date({year}, 3 * quarter - 2, 1) + INTERVAL 3 MONTH AS next_start
                FROM range(1, 5) q(quarter)) q
            WHERE f.founding_date < q.next_start
              AND coalesce(f.exit_date, TIMESTAMP 'infinity') >= q.start_date""")

        feature_dir = ROUTING_DATA / "features" / str(year)
        poi_ok = open_product("destinations", POI_DIR / f"austria-{year}-pois.geoparquet", ["poi_id", "poi_type", "year"], year, "source")
        stale = list(feature_dir.glob("*.partial*")) + list(feature_dir.glob("*.slice*")) + list(feature_dir.glob("*.tmp")) + list(feature_dir.glob(".accessibility_parts*"))
        record(year, "cleanup", "no visible routing intermediates", not stale)
        record(year, "cleanup", "no hidden checkpoint directory", not (WORK_ROOT / str(year)).exists())

        route_types = ["motorway_exit", "regional_centre", "urban_centre", "higher_education"]
        nearest_columns = [f"{prefix}_{kind}{suffix}" for kind in route_types
            for prefix, suffix in [("tt", "_min"), ("km", ""), ("nearest", "_id"), ("routing_status", "")]]
        nearest_ok = open_product("nearest", feature_dir / f"nearest_infrastructure_{RESOLUTION}.parquet", ["grid_id", "year", *nearest_columns], year, "nearest")
        if nearest_ok:
            check_keys("nearest", "expected_nearest", ["grid_id", "year"], year, "nearest")
            for kind in route_types:
                query_ok(year, "nearest", f"{kind} route values match status", f"""SELECT count(*) = 0 FROM nearest
                    WHERE NOT coalesce(
                        (routing_status_{kind} = 'ok' AND isfinite(tt_{kind}_min) AND tt_{kind}_min >= 0
                         AND isfinite(km_{kind}) AND km_{kind} >= 0 AND nearest_{kind}_id IS NOT NULL)
                        OR (routing_status_{kind} IN ('unroutable', 'missing_destinations') AND tt_{kind}_min IS NULL
                            AND km_{kind} IS NULL AND nearest_{kind}_id IS NULL), false)""")
                if poi_ok:
                    query_ok(year, "nearest", f"{kind} selected IDs belong to yearly destinations", f"""SELECT count(*) = 0 FROM nearest n
                        WHERE n.routing_status_{kind} = 'ok' AND NOT EXISTS
                          (SELECT 1 FROM destinations d WHERE CAST(d.poi_id AS VARCHAR) = CAST(n.nearest_{kind}_id AS VARCHAR)
                           AND d.poi_type = {sql_string(kind)} AND d.year = {year})""")

        main_numeric = ["own_cell_pop", "own_cell_firms", *main_access_columns(), *[f"reachable_cells_{m}min" for m in ACCESS_MINUTES]]
        main_ok = open_product("main", feature_dir / f"accessibility_potentials_{RESOLUTION}.parquet", ["grid_id", "year", "quarter", "period", *main_numeric], year, "grid-quarter")
        if main_ok:
            check_keys("main", "expected_grid", ["grid_id", "year", "quarter"], year, "grid-quarter")
            check_period("main", year, "grid-quarter")
            numeric_checks("main", main_numeric, year, "grid-quarter")
            check_contours("main", {p: ACCESS_MINUTES for p in ["pop_access", "existing_firms_access", "reachable_cells"]}, year, "grid-quarter")
            names = [r[0] for r in con.execute("DESCRIBE main").fetchall()]
            record(year, "grid-quarter", "no contemporaneous or Sparte fields", not any(c == "active_firms_t" or "sparte_" in c.lower() for c in names))

        walk_prefixes = ["walk_pop", "walk_firms", "walk_pt_stops", "walk_pt_departures", "walk_pt_routes", "reachable_cells_walk"]
        walk_numeric = ["own_cell_walk_pop", "own_cell_walk_firms", "pt_ohne_haltestelle", *[f"{p}_{m}min" for m in (5, 10) for p in walk_prefixes]]
        walk_ok = open_product("pedestrian", feature_dir / f"pedestrian_accessibility_quarter_{RESOLUTION}.parquet", ["grid_id", "year", "quarter", "period", *walk_numeric], year, "pedestrian-grid-quarter")
        if walk_ok:
            check_keys("pedestrian", "expected_grid", ["grid_id", "year", "quarter"], year, "pedestrian-grid-quarter")
            check_period("pedestrian", year, "pedestrian-grid-quarter")
            numeric_checks("pedestrian", walk_numeric, year, "pedestrian-grid-quarter")
            check_contours("pedestrian", {p: (5, 10) for p in walk_prefixes}, year, "pedestrian-grid-quarter")
            query_ok(year, "pedestrian-grid-quarter", "zero-stop flag matches 10-minute stop count", """SELECT count(*) = 0 FROM pedestrian
                WHERE grid_id NOT IN (SELECT grid_id FROM skipped)
                  AND pt_ohne_haltestelle IS DISTINCT FROM CAST(walk_pt_stops_10min = 0 AS INTEGER)""")

        long_numeric = ["own_cell_same_fachgruppe_firms", "own_cell_walk_same_fachgruppe_firms",
            *[f"same_fachgruppe_firms_access_{m}min" for m in ACCESS_MINUTES], *[f"walk_same_fachgruppe_firms_{m}min" for m in (5, 10)]]
        long_ok = open_product("fachgruppe", feature_dir / f"fachgruppe_accessibility_quarter_{RESOLUTION}.parquet", ["grid_id", "year", "quarter", "Fachgruppe_ID", *long_numeric], year, "Fachgruppe-long")
        if long_ok:
            query_ok(year, "Fachgruppe-long", "only configured Fachgruppe IDs", "SELECT count(*) = 0 FROM fachgruppe WHERE Fachgruppe_ID IS NULL OR CAST(Fachgruppe_ID AS VARCHAR) NOT IN (SELECT Fachgruppe_ID FROM configured_groups)")
            # Check one partition at a time: no 48-million-row distinct/join in memory.
            for group in FACHGRUPPE_IDS:
                con.execute(f"CREATE OR REPLACE TEMP VIEW one_group AS SELECT * FROM fachgruppe WHERE CAST(Fachgruppe_ID AS VARCHAR) = {sql_string(group)}")
                check_keys("one_group", "expected_grid", ["grid_id", "year", "quarter"], year, f"Fachgruppe {group}")
            numeric_checks("fachgruppe", long_numeric, year, "Fachgruppe-long")
            check_contours("fachgruppe", {"same_fachgruppe_firms_access": ACCESS_MINUTES, "walk_same_fachgruppe_firms": (5, 10)}, year, "Fachgruppe-long")

        firm_numeric = ["own_cell_pop", "own_cell_firms", "own_cell_walk_pop", "own_cell_walk_firms", *main_access_columns(), *long_numeric]
        firm_ok = open_product("firm", feature_dir / f"firm_accessibility_quarter_{RESOLUTION}.parquet", ["firm_id", "grid_id", "Fachgruppe_ID", "year", "quarter", "period", "included_in_lagged_stock", *firm_numeric], year, "firm-quarter")
        if firm_ok:
            check_keys("firm", "expected_firms", ["firm_id", "grid_id", "year", "quarter", "Fachgruppe_ID", "included_in_lagged_stock"], year, "firm-quarter")
            # Including metadata in coverage above also detects altered grids/groups/stock flags.
            query_ok(year, "firm-quarter", "unique firm-year-quarter keys", "SELECT count(*) = count(DISTINCT (firm_id, year, quarter)) FROM firm")
            check_period("firm", year, "firm-quarter")
            numeric_checks("firm", list(dict.fromkeys(firm_numeric)), year, "firm-quarter")
            query_ok(year, "firm-quarter", "configured Fachgruppe coverage at least 95%", "SELECT count(*) = 0 OR count(*) FILTER (WHERE CAST(Fachgruppe_ID AS VARCHAR) IN (SELECT Fachgruppe_ID FROM configured_groups)) >= .95 * count(*) FROM firm")
            if long_ok and main_ok:
                # Restrict the long reference first; do not build a huge right-hand join table.
                con.execute("""CREATE OR REPLACE TEMP TABLE firm_reference AS SELECT l.* FROM fachgruppe l
                    SEMI JOIN firm f ON l.grid_id=f.grid_id AND l.year=f.year AND l.quarter=f.quarter
                        AND CAST(l.Fachgruppe_ID AS VARCHAR)=CAST(f.Fachgruppe_ID AS VARCHAR)""")
                comparisons = [f'f."{c}" IS NULL OR r."{c}" IS NULL OR abs(f."{c}" - r."{c}") > 1e-8 + 1e-5 * abs(r."{c}")' for c in long_numeric]
                query_ok(year, "firm-quarter", "known-Fachgruppe rows all match long table and masses", f"""SELECT count(*) = 0 FROM firm f
                    LEFT JOIN firm_reference r ON f.grid_id=r.grid_id AND f.year=r.year AND f.quarter=r.quarter
                        AND CAST(f.Fachgruppe_ID AS VARCHAR)=CAST(r.Fachgruppe_ID AS VARCHAR)
                    WHERE f.grid_id NOT IN (SELECT grid_id FROM skipped)
                      AND CAST(f.Fachgruppe_ID AS VARCHAR) IN (SELECT Fachgruppe_ID FROM configured_groups)
                      AND (r.grid_id IS NULL OR {' OR '.join(comparisons)})""")
                fields = ["own_cell_pop", "own_cell_firms", *main_access_columns()]
                comparisons = [f'f."{c}" IS NULL OR m."{c}" IS NULL OR abs(f."{c}" - m."{c}") > 1e-8 + 1e-5 * abs(m."{c}")' for c in fields]
                query_ok(year, "firm-quarter", "all rows match grid table and lagged masses", f"""SELECT count(*) = 0 FROM firm f
                    LEFT JOIN main m ON f.grid_id=m.grid_id AND f.year=m.year AND f.quarter=m.quarter
                    WHERE f.grid_id NOT IN (SELECT grid_id FROM skipped)
                      AND (m.grid_id IS NULL OR {' OR '.join(comparisons)})""")

    if STATUS_PATH.exists():
        status = pd.read_csv(STATUS_PATH)
        fields = {"year", "status", "firm_mass_source"}
        if record("all", "manifest", "routing status schema", fields.issubset(status)):
            status = status.drop_duplicates("year", keep="last")
            done_years = set(pd.to_numeric(status.loc[status.status.eq("done"), "year"], errors="coerce").dropna().astype(int))
            record("all", "manifest", "all requested years completed", done_years >= set(YEARS))
            record("all", "manifest", "model products use lagged firm stock", status.firm_mass_source.notna().all() and set(status.firm_mass_source) == {"active_firms_tminus1"})
    else:
        record("all", "manifest", "routing status exists", False)
    record("all", "manifest", "no unresolved failed-origin manifest", not list(WORK_ROOT.glob("*/failed_origins.json")))

    if GRAPH_STATUS_PATH.exists():
        graphs = pd.read_csv(GRAPH_STATUS_PATH).drop_duplicates("year", keep="last")
        fields = {"year", "status", "manifest_path_wsl", "valhalla_image", "valhalla_image_id", "gip_snapshot_sha256"}
        if record("all", "graphs", "graph status schema", fields.issubset(graphs)):
            done_years = set(pd.to_numeric(graphs.loc[graphs.status.eq("done"), "year"], errors="coerce").dropna().astype(int))
            record("all", "graphs", "successful graph manifests for requested years", done_years >= set(YEARS) and graphs.manifest_path_wsl.notna().all())
            record("all", "graphs", "pinned graph-builder image recorded", graphs.valhalla_image_id.notna().all() and graphs.valhalla_image.notna().all() and set(graphs.valhalla_image_id) == {GRAPH_VALHALLA_IMAGE_ID} and set(graphs.valhalla_image) == {GRAPH_VALHALLA_IMAGE})
            actual = graphs.set_index("year").gip_snapshot_sha256.to_dict()
            for year in YEARS:
                path = PBF_DIR / f"{year}.osm.pbf"
                record(year, "graphs", "graph manifest matches source PBF hash", path.exists() and actual.get(year) == sha256_file(path))
    else:
        record("all", "graphs", "graph status exists", False)
finally:
    con.close()
    validation = pd.DataFrame(records)
    validation.to_csv(REPORT_PATH, index=False)

summary = validation.groupby(["product", "status"]).size().rename("checks").reset_index()
if validation.status.ne("OK").any():
    raise AssertionError("Routing validation failed; no analysis should start.\n" + validation.loc[validation.status.ne("OK")].to_string(index=False))
summary
